# Neuron Interaction Analysis

Analyze how MLP neurons interact: coactivation patterns, interference via
shared output directions, cooperative groups, compensation when ablated,
and ensemble synergy/redundancy.

## Why This Matters

MLP neuron interaction provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.neuron_interaction_analysis import (
    neuron_coactivation_matrix,
    neuron_interference,
    cooperative_neuron_groups,
    neuron_compensation,
    neuron_ensemble_effect,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Neuron Coactivation Matrix

Which neurons fire together? Compute pairwise coactivation rates.

In [ ]:
result = neuron_coactivation_matrix(model, tokens, layer=0)
print(f"Neurons analyzed: {result['n_neurons_analyzed']}")
print(f"Mean coactivation: {result['mean_coactivation']:.4f}")
print(f"Most coactivated pair: neurons {result['max_coactivated_pair']} "
      f"(rate={result['max_coactivation']:.4f})")

## Neuron Interference

Measure how much neurons interfere with each other through shared W_out directions.

In [ ]:
result = neuron_interference(model, tokens, layer=0, top_k=5)
print(f"Most interfered neuron: {result['most_interfered']['neuron_idx']}\n")
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron_idx']:3d}: activation={n['activation']:.4f}, "
          f"interference={n['total_interference']:.4f}, "
          f"n_interfering={n['n_interfering']}")

## Cooperative Neuron Groups

Find clusters of neurons that reliably activate together.

In [ ]:
result = cooperative_neuron_groups(model, tokens, layer=0, threshold=0.3)
print(f"Found {result['n_groups']} cooperative groups")
print(f"Mean correlation: {result['mean_correlation']:.4f}")
for i, group in enumerate(result['groups'][:5]):
    print(f"  Group {i}: neurons {group}")

## Neuron Compensation

When one neuron is ablated, do others compensate?

In [ ]:
for neuron_idx in [0, 5, 10]:
    result = neuron_compensation(model, tokens, layer=0, neuron_idx=neuron_idx)
    print(f"Neuron {result['neuron_idx']:3d}: activation={result['neuron_activation']:.4f}, "
          f"logit_effect={result['mean_logit_effect']:.4f}, "
          f"compensation={result['compensation_ratio']:.2%}, "
          f"preds_changed={result['predictions_changed']}")

## Ensemble Effect

Is the joint effect of ablating multiple neurons more or less than the sum of individual effects?

In [ ]:
result = neuron_ensemble_effect(model, tokens, layer=0, top_k=5)
print(f"Sum of individual effects: {result['sum_individual']:.4f}")
print(f"Joint effect:              {result['joint_effect']:.4f}")
print(f"Synergy ratio:             {result['synergy_ratio']:.4f}")
print(f"Synergistic: {result['is_synergistic']}, Redundant: {result['is_redundant']}")
print(f'\nIndividual effects:')
for e in result['individual_effects']:
    print(f"  Neuron {e['neuron_idx']:3d}: {e['individual_effect']:.4f}")